In [ ]:
# Importing required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline
import seaborn as sns
import scipy.stats as si
import statsmodels.api as sm
import statsmodels.tsa.api as tsa
from scipy.stats import zscore
from statsmodels.tsa.ar_model import AutoReg
from scipy import stats as st
from scipy.optimize import least_squares
from tabulate import tabulate
import warnings
warnings.filterwarnings("ignore")
from scipy.stats.mstats import gmean
import statsmodels.formula.api as smf
import itertools
from itertools import product

In [ ]:
import wrds
conn = wrds.Connection()

# Factor Momentum, Reversal, and Turning Points

본 논문에서는, 여러 factor 기반 portolios의 수익률 변화에 따른 momentum, reversal, 그리고 그 변곡점인 turning points에 대한 조사를 실시한다.   

Factor portfolios는 market dynamics에 따라 수익률이 변화하며, multi factor model의 factor exposure 조절에 따라 dynamic한 strategy를 수립할 수 있다.

## Introduction

추후 추가

## Literature Review 

추후 추가

## Data

Data는 153개 factor portfolio에 대해 보고하는 Jensen, Kelly, and Pedersen (2023) 데이터를 사용한다. 해당 datset은 CRSP, Compustat data를 사용해 여러 factor portolios를 제작하고 수익률을 보고하는 것은로 자세한 사항은 jkpfactors.com에서 보고한다.

우리는 이 1972-01부터 2023-12 까지 50년을 조사한다.

Figure 1 에서 data에 대한 summary를 보고한다.

In [ ]:
# Factor Theme
usa_th_ew = pd.read_csv("C:/Users/YeonChan Kang/Desktop/Local_repo/Factor-Momentum-Reversal-and-Turning-Point/data/themes/[usa]_[all_themes]_[monthly]_[ew].csv")
usa_th_vw = pd.read_csv("C:/Users/YeonChan Kang/Desktop/Local_repo/Factor-Momentum-Reversal-and-Turning-Point/data/themes/[usa]_[all_themes]_[monthly]_[vw].csv")
usa_th_cvw = pd.read_csv("C:/Users/YeonChan Kang/Desktop/Local_repo/Factor-Momentum-Reversal-and-Turning-Point/data/themes/[usa]_[all_themes]_[monthly]_[vw_cap].csv")

# Factor
usa_fac_ew = pd.read_csv("C:/Users/YeonChan Kang/Desktop/Local_repo/Factor-Momentum-Reversal-and-Turning-Point/data/factors/[usa]_[all_factors]_[monthly]_[ew].csv")
usa_fac_vw = pd.read_csv("C:/Users/YeonChan Kang/Desktop/Local_repo/Factor-Momentum-Reversal-and-Turning-Point/data/factors/[usa]_[all_factors]_[monthly]_[vw].csv")
usa_fac_cvw = pd.read_csv("C:/Users/YeonChan Kang/Desktop/Local_repo/Factor-Momentum-Reversal-and-Turning-Point/data/factors/[usa]_[all_factors]_[monthly]_[vw_cap].csv")

In [ ]:
# Manipulating the format of the data
usa_fac_ew.drop(columns = ['freq', 'weighting', 'location','n_stocks', 'n_stocks_min', 'direction'], inplace = True)
usa_fac_ew = usa_fac_ew.pivot_table(index='date', columns='name', values='ret')

usa_fac_vw.drop(columns = ['freq', 'weighting', 'location','n_stocks', 'n_stocks_min', 'direction'], inplace = True)
usa_fac_vw = usa_fac_vw.pivot_table(index='date', columns='name', values='ret')

usa_fac_cvw.drop(columns = ['freq', 'weighting', 'location','n_stocks', 'n_stocks_min', 'direction'], inplace = True)
usa_fac_cvw = usa_fac_cvw.pivot_table(index='date', columns='name', values='ret')

In [ ]:
usa_fac_ew.index = pd.to_datetime(usa_fac_ew.index)
usa_fac_ew.index = usa_fac_ew.index.to_period('M')

usa_fac_cvw.index = pd.to_datetime(usa_fac_cvw.index)
usa_fac_cvw.index = usa_fac_cvw.index.to_period('M')

usa_fac_vw.index = pd.to_datetime(usa_fac_vw.index)
usa_fac_vw.index = usa_fac_vw.index.to_period('M')

In [ ]:
ff_data = conn.raw_sql("""
                       SELECT *
                       FROM ff.factors_monthly
                       WHERE date between '01/01/1926' and '12/31/2023'
                       """)

In [ ]:
ff_data.drop(columns=['date'], inplace=True)
ff_data.index.name = 'date'
ff_data.set_index('dateff', inplace=True)
ff_data.index = pd.to_datetime(ff_data.index)
ff_data.index = ff_data.index.to_period('M')

In [ ]:
usa_fac_ewm = pd.merge(ff_data[['mktrf']], usa_fac_ew, left_index=True, right_index=True, how='outer')

usa_fac_vwm = pd.merge(ff_data[['mktrf']], usa_fac_vw, left_index=True, right_index=True, how='outer')

usa_fac_cvwm = pd.merge(ff_data[['mktrf']], usa_fac_cvw, left_index=True, right_index=True, how='outer')

In [ ]:
usa_fac_ewm = usa_fac_ewm[usa_fac_ewm.index >= '1972-01']
usa_fac_vwm = usa_fac_vwm[usa_fac_vwm.index >= '1972-01']
usa_fac_cvwm = usa_fac_cvwm[usa_fac_cvwm.index >= '1972-01']

## Methodology

## Empirical evidence

### Factor momentum and Reversal

Factor momentum에 대해 조사한다.
우리의 기준은 각각, 
- 직전 1개월(1-1), 
- 직전 1개월부터 3개월(1-3), 
- 직전 1개월 부터 6개월(1-6),
- 직전 1개월부터 12개월(1-12), 
- 직전 1개월부터 60개월(1-60)과
전통적인 cross-sectional momenumt기준인 
- 2개월 부터 12개월 (2-12),
- 7개월부터 12개월 (7-12), 
- 직전 1년을 제외한 (13-60), 
- long-term reversal로 측정되는 (37-60)
을 조사한다.


해당 기간 동안, 수익률은 다음과 같다.

In [ ]:
# Creating the lagged and rolling variables

# (1-1)
usa_fac_cvwm_r1_1 = usa_fac_cvwm.copy()
for col in usa_fac_cvwm.columns:
    usa_fac_cvwm_r1_1[col] = usa_fac_cvwm[col].shift(1).astype(float)

# (1-3)
usa_fac_cvwm_r1_3 = usa_fac_cvwm.copy()
for col in usa_fac_cvwm.columns:
    usa_fac_cvwm_r1_3[col] = usa_fac_cvwm[col].shift(1).rolling(3).mean().astype(float)

# (1-6)
usa_fac_cvwm_r1_6 = usa_fac_cvwm.copy()
for col in usa_fac_cvwm.columns:
    usa_fac_cvwm_r1_6[col] = usa_fac_cvwm[col].shift(1).rolling(6).mean().astype(float)

# (1-12)
usa_fac_cvwm_r1_12 = usa_fac_cvwm.copy()
for col in usa_fac_cvwm.columns:
    usa_fac_cvwm_r1_12[col] = usa_fac_cvwm[col].shift(1).rolling(12).mean().astype(float)

# (1-60)
usa_fac_cvwm_r1_60 = usa_fac_cvwm.copy()
for col in usa_fac_cvwm.columns:
    usa_fac_cvwm_r1_60[col] = usa_fac_cvwm[col].shift(1).rolling(60).mean().astype(float)

# (2-12)
usa_fac_cvwm_r2_12 = usa_fac_cvwm.copy()
for col in usa_fac_cvwm.columns:
    usa_fac_cvwm_r2_12[col] = usa_fac_cvwm[col].shift(2).rolling(11).mean().astype(float)

# (7-12)
usa_fac_cvwm_r7_12 = usa_fac_cvwm.copy()
for col in usa_fac_cvwm.columns:
    usa_fac_cvwm_r7_12[col] = usa_fac_cvwm[col].shift(7).rolling(6).mean().astype(float)

# (37-60)
usa_fac_cvwm_r37_60 = usa_fac_cvwm.copy()
for col in usa_fac_cvwm.columns:
    usa_fac_cvwm_r37_60[col] = usa_fac_cvwm[col].shift(37).rolling(24).mean().astype(float)

#1 Cross-sectional Factor momentum

In [201]:
def csfm(df, startdate, enddate):
    # 새로운 데이터프레임 생성 (원본 데이터프레임과 동일한 구조)
    cumulative_returns = pd.DataFrame(index=df.index, columns=df.columns)
    
    for col in df.columns:
        # 각 날짜에 대해 누적 수익률을 계산
        cumulative_returns[col] = df[col].rolling(enddate - startdate+1).sum().shift(startdate -1)

    return cumulative_returns

In [208]:
usa_fac_cvwm_m1_1 =csfm(usa_fac_cvwm, 1, 1)
usa_fac_cvwm_m1_3 =csfm(usa_fac_cvwm, 1, 3)
usa_fac_cvwm_m1_6 =csfm(usa_fac_cvwm, 1, 6)
usa_fac_cvwm_m1_12 =csfm(usa_fac_cvwm, 1, 12)
usa_fac_cvwm_m1_60 =csfm(usa_fac_cvwm, 1, 60)
usa_fac_cvwm_m2_12 =csfm(usa_fac_cvwm, 2, 12)
usa_fac_cvwm_m7_12 =csfm(usa_fac_cvwm, 7, 12)
usa_fac_cvwm_m37_60 =csfm(usa_fac_cvwm, 37, 60)

In [221]:
usa_fac_cvwm_m2_12.loc['2000-01'].sort_values(ascending=False).head(10)

rd_sale         0.791290
rd5_at          0.563170
cash_at         0.530221
bidaskhl_21d    0.505221
netdebt_me      0.459300
age             0.441496
at_be           0.380265
tangibility     0.365615
seas_1_1na      0.353462
ret_12_1        0.350134
Name: 2000-01, dtype: float64

In [222]:
usa_fac_cvwm_m2_12.loc['2000-01'].sort_values(ascending=True).head(10)

ivol_capm_252d   -0.564712
rvol_21d         -0.531516
rmax5_21d        -0.508287
ivol_hxz4_21d    -0.496694
at_me            -0.494272
ivol_ff3_21d     -0.492626
ni_me            -0.489825
ivol_capm_21d    -0.487936
rmax1_21d        -0.485708
sale_me          -0.484010
Name: 2000-01, dtype: float64

In [240]:
def sign_csfm(df, sdate, edate, longshort=5):
    
    momentum = csfm(df, sdate, edate)
    
    l_col = [f'l{i}' for i in range(1, longshort+1)]
    s_col = [f's{i}'for i in range(1, longshort+1)]
    sign_df = pd.DataFrame(index=df.index, columns=l_col+s_col)
    
    for idx in df.index:
        sign_df.loc[idx][l_col] = momentum.loc[idx].sort_values(ascending=False).head(longshort).index
        sign_df.loc[idx][s_col] = momentum.loc[idx].sort_values(ascending=True).head(longshort).index
    
    return sign_df

In [225]:
tset_df = sign_csfm(usa_fac_cvwm, 2, 12, 10)

In [237]:
tset_df.loc['2000-01'].str.contains('l')

l1      True
l2     False
l3     False
l4      True
l5     False
l6     False
l7     False
l8      True
l9     False
l10    False
s1      True
s2      True
s3     False
s4      True
s5     False
s6      True
s7     False
s8      True
s9     False
s10     True
Name: 2000-01, dtype: bool

In [239]:
tset_df

,l1,l2,l3,l4,l5,l6,l7,l8,l9,l10,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10
1972-01,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a
1972-02,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a
1972-03,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a
1972-04,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a
1972-05,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08,seas_16_20an,netis_at,gp_at,mktrf,op_at,gp_atl1,op_atl1,cop_at,ocf_at,cop_atl1,betabab_1260d,corr_1260d,beta_dimson_21d,betadown_252d,beta_60m,inv_gr1a,saleq_su,ami_126d,market_equity,seas_11_15na
2023-09,mktrf,op_at,op_atl1,netis_at,gp_at,at_turnover,gp_atl1,cop_at,ocf_at,cop_atl1,corr_1260d,betabab_1260d,beta_dimson_21d,beta_60m,betadown_252d,ami_126d,market_equity,inv_gr1a,dolvol_126d,ret_3_1
2023-10,seas_16_20an,op_atl1,op_at,gp_at,gp_atl1,cop_at,cop_atl1,at_turnover,ocf_at,netis_at,corr_1260d,betabab_1260d,ret_9_1,ami_126d,market_equity,betadown_252d,beta_dimson_21d,ret_3_1,inv_gr1a,dolvol_126d
2023-11,seas_16_20an,gp_atl1,cop_atl1,op_atl1,gp_at,cop_at,ocf_at,op_at,sale_bev,at_turnover,ret_9_1,corr_1260d,ret_60_12,market_equity,inv_gr1a,ami_126d,emp_gr1,seas_11_15an,sale_gr3,dolvol_126d


In [232]:
def cal_momport(df, sign_df):
    
    momport = pd.DataFrame(index=df.index, columns=['l', 's'])
    
    for idx in df.index:
        momport.loc[idx]['l'] = df.loc[idx][sign_df.loc[idx].str.contains('l')]
        momport.loc[idx]['s'] = df.loc[idx][sign_df.loc[idx].str.contains('s')]
    
    return momport

In [233]:
cal_momport(usa_fac_cvwm, tset_df)

KeyError: "None of [Index(['mktrf', 'age', 'aliq_at', 'aliq_mat', 'ami_126d', 'at_be', 'at_gr1',\n       'at_me', 'at_turnover', 'be_gr1a', 'mktrf', 'age', 'aliq_at',\n       'aliq_mat', 'ami_126d', 'at_be', 'at_gr1', 'at_me', 'at_turnover',\n       'be_gr1a'],\n      dtype='object')] are in the [columns]"

In [211]:
usa_fac_cvwm_m37_60.head(61)

,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a,...,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,z_score,zero_trades_126d,zero_trades_21d,zero_trades_252d
1972-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1972-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1972-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1972-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1972-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1976-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1976-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1976-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1976-12,-0.1446,-0.253151,0.195857,-0.066443,-0.23255,0.080816,0.126694,-0.023985,-0.281539,0.167238,...,0.146608,0.194551,0.302984,-0.088213,0.291062,0.145151,-0.045602,0.229363,0.102545,0.208930


In [207]:
csfm(usa_fac_cvwm, 1, 12)

,mktrf,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,be_gr1a,...,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,z_score,zero_trades_126d,zero_trades_21d,zero_trades_252d
1972-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1972-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1972-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1972-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1972-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08,0.1149,-0.096439,-0.002573,0.010635,-0.125592,0.066004,0.037247,-0.044664,0.153550,-0.033544,...,0.027909,-0.026757,0.008925,0.079469,-0.022366,0.082763,0.068448,-0.023906,-0.030800,-0.031611
2023-09,0.1560,-0.104356,-0.002113,-0.026269,-0.153208,0.055387,0.037505,-0.011016,0.174440,-0.045426,...,0.014529,-0.032639,0.016958,0.097224,-0.034203,0.105341,0.051860,-0.035190,-0.038030,-0.042948
2023-10,0.0459,-0.094843,-0.008939,0.018319,-0.160758,0.072017,-0.005609,-0.054758,0.153423,-0.051190,...,0.025190,-0.016895,0.006853,0.074567,-0.008842,0.081729,0.092502,-0.012888,-0.013168,-0.014897
2023-11,0.0883,0.023285,-0.056591,0.027830,-0.102514,0.046989,-0.045336,-0.069573,0.108438,-0.039836,...,0.050594,0.011461,0.017429,0.054607,-0.061441,0.034451,0.082089,-0.064826,-0.048863,-0.071683


In [206]:
usa_fac_cvwm['mktrf'].iloc[0:12].sum()

0.1216

In [200]:
usa_fac_cvwm['mktrf'].iloc[1:12].sum()

0.0967

In [158]:
usa_fac_cvwm.index[0]

Period('1972-01', 'M')

In [180]:
usa_fac_cvwm.index[10]

Period('1972-11', 'M')

In [181]:
usa_fac_cvwm['mktrf'].iloc[0:10].sum()

0.0694

In [162]:
usa_fac_cvwm.index[1]

Period('1972-02', 'M')

In [163]:
usa_fac_cvwm.index[11]

Period('1972-12', 'M')

In [164]:
usa_fac_cvwm['mktrf'].iloc[1:11].sum()

0.0905

In [121]:
def csfm(df, factors, start, end):
    df_m = df.copy()
    
    df_m = df_m.shift(start).rolling(end).sum()
    
    return df_m

In [136]:
# 0.1216	


0.06799999999999999

In [111]:
usa_fac_cvwm['mktrf'].iloc[1:12].sum()

0.0967

In [ ]:
csfm(usa_fac_cvwm, usa_fac_cvwm.columns, 1, 1)

In [72]:
csfm(usa_fac_cvwm, usa_fac_cvwm.columns, 2, 11)['mktrf'][13]

0.0967

In [78]:
csfm(usa_fac_cvwm, usa_fac_cvwm.columns, 2, 11).index[13]

Period('1973-02', 'M')

In [99]:
csfm(usa_fac_cvwm, usa_fac_cvwm.columns, 2, 13)['mktrf'][14]

0.0887

In [ ]:
csfm(usa_fac_cvwm, usa_fac_cvwm.columns, 2, 13)['mktrf'][14]

In [89]:
usa_fac_cvwm['mktrf'][1:13].sum()

0.0638

In [96]:
usa_fac_cvwm['mktrf'][2:13].sum()

0.03509999999999999

In [101]:
usa_fac_cvwm.index[12]

Period('1973-01', 'M')

#2 Time-series Factor momentum